# Combo Strategy v2 — Reversal Signal Scanner

**Mục tiêu:** Scan tín hiệu reversal trong N bar H4 gần nhất — khi có tín hiệu ngược chiều sẽ đóng lệnh cũ, mở lệnh mới ngay lập tức.

**Rules áp dụng (Combo v2 — Reversal):**
- 3 điều kiện: Close vs Open / MA crossover / MACD(5,25,5)
- Session filter: chỉ bar H4 trong giờ giao dịch của từng sàn
- R:R filter: chỉ trade khi R:R ≥ MIN_RR
- **Reversal logic**: signal ngược chiều → đóng lệnh cũ + mở mới (outcome = `Reversed`)
- US30 Macro filter: US100/US500 chỉ cùng chiều MA(20) của US30

Chạy từng cell theo thứ tự `Shift+Enter`

In [1]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import warnings

warnings.filterwarnings('ignore')

import sys

# ── Bootstrap: add project root to sys.path ──────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd

_ROOT = Path().resolve().parents[3]   # research/ → combo/ → strategies/ → core_python/ → project root
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from modules.db_connector import get_connection, test_connection
from strategies.combo.core.scan_pipeline import (
    calc_reversal_stats,
    prepare_data,
    run_multi_reversal_scan,
    run_reversal_scan,
)
from strategies.combo.core.strategy_config import (
    DEFAULT_N_BARS,
    INDICATOR_COLS,
    STRATEGY,
    SYMBOLS,
    TIMEFRAME,
    US30_KEY,
    US30_SYMBOL_ID,
    US_FILTERED,
    get_indicator_params,
)
from strategies.combo.core.strategy_config import (
    summary as strategy_summary,
)
from strategies.combo.core.theme import (
    DARK,
    EQUITY_COLORS,
    NUM_FMT,
    SIGNAL,
    setup_dark_axes,
    setup_dark_figure,
    style_reversal_row,
)

print(strategy_summary())

Strategy     : Combo v2
Symbols      : US30, HK50, J225
US30 filter  : none
MA period    : 20
MACD         : (5, 25, 5)
ATR period   : 5
kTP          : 2.3
Min R:R      : 1.25


In [2]:
# ── Cell 2: SCAN OPTIONS — chỉnh tại đây ─────────────────────────────────────
# Thông số chiến lược (kTP, x, MA, ...) đã được load từ strategy_config.py
# Chỉ cần chỉnh các tuỳ chọn scan & chart dưới đây.

SCAN_SYMBOL = 'US30'   # Đổi thành bất kỳ key trong strategy_config.SYMBOLS
N_BARS      = 100      # Số bar H4 gần nhất để scan và vẽ chart

# ── Chart options ─────────────────────────────────────────────────────────────
SHOW_REJECTED_SIGNALS = True   # Hiển thị tín hiệu bị loại (R:R < min_rr) bằng màu mờ
SHOW_MA_LINE          = True   # Hiển thị đường MA(20)
SHOW_ENTRY_LINES      = True   # Vẽ đường Entry / SL / TP cho mỗi tín hiệu
ENTRY_LINE_BARS       = 8      # Số bar kéo dài đường Entry/SL/TP sang phải

# ── Load strategy params (auto từ strategy_config) ───────────────────────────
P      = get_indicator_params()
KTP    = P['KTP']
MIN_RR = P['MIN_RR']
MA_PERIOD   = P['MA_PERIOD']
ATR_PERIOD  = P['ATR_PERIOD']
MACD_FAST   = P['MACD_FAST']
MACD_SLOW   = P['MACD_SLOW']
MACD_SIGNAL = P['MACD_SIGNAL']

print(f'Scan target  : {SCAN_SYMBOL}  |  N={N_BARS} bars')
print(f'Strategy     : kTP={KTP}  Min R:R={MIN_RR}  MA={MA_PERIOD}  ATR={ATR_PERIOD}')
print(f'MACD         : ({MACD_FAST},{MACD_SLOW},{MACD_SIGNAL})')
print('Mode         : REVERSAL (signal ngược chiều → đóng lệnh cũ + mở mới)')

Scan target  : US30  |  N=100 bars
Strategy     : kTP=2.3  Min R:R=1.25  MA=20  ATR=5
MACD         : (5,25,5)
Mode         : REVERSAL (signal ngược chiều → đóng lệnh cũ + mở mới)


In [3]:
# ── Cell 3: Symbol Config — từ strategy_config.py ────────────────────────────
TARGET_SYMBOLS      = SYMBOLS
US_FILTERED_SYMBOLS = US_FILTERED

cfg_df = pd.DataFrame(TARGET_SYMBOLS).T[['label','session_hours_utc','x','us_macro_filter']]
cfg_df.index.name = 'symbol'
display(cfg_df)

,label,session_hours_utc,x,us_macro_filter
symbol,,,,
US30,US30 (Dow Jones),[],13.0,False
HK50,HK50 (Hang Seng 50),[],8.0,False
J225,J225 (Nikkei 225),[],8.0,False


In [4]:
# ── Cell 4: DB Connection ─────────────────────────────────────────────────────
ok = test_connection()
print('✓ SQL Server connected' if ok else '✗ Connection FAILED')

[OK] SQL Server connected. 20 tables found in SEN/DWH/MART.
✓ SQL Server connected


In [5]:
# ── Cell 5: Shared modules (loaded via scan_pipeline + theme in Cell 1) ──────
from modules.chart_builder import build_reversal_chart

print('✓ All modules loaded: reversal_scanner + scan_pipeline + theme + chart_builder')

✓ All modules loaded: reversal_scanner + scan_pipeline + theme + chart_builder


In [6]:
# ── Cell 6: Load data + reversal scan ─────────────────────────────────────────
result     = run_reversal_scan(SCAN_SYMBOL, N_BARS, P)
df_scan    = result['df_scan']
signals_df = result['signals_df']
cfg        = result['cfg']

# ── Summary ──────────────────────────────────────────────────────────────────
stats = calc_reversal_stats(signals_df)

print(f'\n── Reversal Scan: {SCAN_SYMBOL}  ({N_BARS} bars) ──')
print(f'  Total raw signals      : {stats["n_total"]}')
print(f'  Reversal signals       : {stats["n_reversal_signals"]}')
print(f'  Pass R:R ≥ {MIN_RR}      : {stats["n_pass"]}  ({100*stats["n_pass"]//max(stats["n_total"],1)}%)')
print(f'  Rejected (R:R < {MIN_RR}) : {stats["n_rejected"]}')
print(f'  BUY signals            : {stats["n_buy"]}')
print(f'  SELL signals           : {stats["n_sell"]}')
if stats['n_total'] > 0:
    print(f'  Avg R:R (all)          : {signals_df["rr"].mean():.2f}')
    if stats['n_pass'] > 0:
        print(f'  Avg R:R (passed)       : {stats["avg_rr"]:.2f}')

if stats['n_pass'] > 0:
    print('\n  ── Trade outcomes (pass R:R only) ──')
    print(f'  Hit TP      ✅ : {stats["n_tp"]}  ({stats["win_pct"]:.0f}%)')
    print(f'  Hit SL      ❌ : {stats["n_sl"]}')
    print(f'  Đảo chiều   🔄 : {stats["n_reversed"]}')
    if stats['n_open']:
        print(f'  Still open     : {stats["n_open"]}  (SL/TP not reached in {N_BARS} bars)')

[data_loader] ⚠️  H4: phat hien 384 nen bi thieu

── Reversal Scan: US30  (100 bars) ──
  Total raw signals      : 1
  Reversal signals       : 0
  Pass R:R ≥ 1.25      : 1  (100%)
  Rejected (R:R < 1.25) : 0
  BUY signals            : 0
  SELL signals           : 1
  Avg R:R (all)          : 11.35
  Avg R:R (passed)       : 11.35

  ── Trade outcomes (pass R:R only) ──
  Hit TP      ✅ : 0  (0%)
  Hit SL      ❌ : 1
  Đảo chiều   🔄 : 0


In [7]:
# ── Cell 7: Bảng tín hiệu ────────────────────────────────────────────────────
# Thêm cột Result (emoji) và P&L (points) để thấy rõ kết quả từng lệnh

_OUTCOME_EMOJI = {'TP': '✅ TP', 'SL': '❌ SL', 'Reversed': '🔄 Đảo chiều',
                  'Open': '⏳ Open'}

def _add_result_pnl(df):
    """Thêm cột result (emoji) và pnl_pts vào DataFrame tín hiệu."""
    df['result'] = df['outcome'].map(_OUTCOME_EMOJI).fillna('— Rejected')
    def _calc_pnl(r):
        if r['outcome'] == 'TP':
            return round(r['tp_dist'], 1)
        elif r['outcome'] == 'SL':
            return round(-r['sl_dist'], 1)
        elif r['outcome'] == 'Reversed':
            return round(-r['sl_dist'] * 0.5, 1)
        return 0.0
    df['pnl_pts'] = df.apply(_calc_pnl, axis=1)
    return df


if signals_df.empty:
    print('Không có tín hiệu nào trong khoảng này.')
else:
    tbl = signals_df.copy()
    tbl = _add_result_pnl(tbl)

    display_cols = ['bar_time','direction','is_reversal','result','pnl_pts',
                    'entry','sl','tp','rr','pass_rr','atr','sl_dist','tp_dist']
    tbl = tbl[display_cols]
    tbl['bar_time'] = tbl['bar_time'].dt.strftime('%Y-%m-%d %H:%M')

    styled = (tbl.style
        .apply(style_reversal_row, axis=1)
        .map(lambda v: (f'color:{SIGNAL["tp"]};font-weight:bold' if 'TP' in str(v)
                        else (f'color:{SIGNAL["sl"]};font-weight:bold' if 'SL' in str(v)
                              else ('color:#FFB347;font-weight:bold' if 'Đảo' in str(v)
                                    else ''))),
             subset=['result'])
        .map(lambda v: (f'color:{SIGNAL["tp"]};font-weight:bold' if v > 0
                        else (f'color:{SIGNAL["sl"]};font-weight:bold' if v < 0
                              else 'color:#555')),
             subset=['pnl_pts'])
        .format({**NUM_FMT, 'pnl_pts': '{:+.1f}'})
        .set_caption(
            f'Reversal Scanner — {SCAN_SYMBOL}  |  '
            f'✅ TP  ❌ SL  🔄 Đảo chiều  Min R:R={MIN_RR}'
        )
    )
    display(styled)

    # ── Tóm tắt P&L ──────────────────────────────────────────────────────
    passed = tbl[tbl['pass_rr']]
    if not passed.empty:
        total_pnl = passed['pnl_pts'].sum()
        tp_pnl    = passed[passed['pnl_pts'] > 0]['pnl_pts'].sum()
        sl_pnl    = passed[passed['pnl_pts'] < 0]['pnl_pts'].sum()
        print('\n  P&L Summary (passed signals only):')
        print(f'    ✅ TP total : +{tp_pnl:.1f} pts')
        print(f'    ❌ SL total : {sl_pnl:.1f} pts')
        print(f'    Net P&L    : {total_pnl:+.1f} pts')

,bar_time,direction,is_reversal,result,pnl_pts,entry,sl,tp,rr,pass_rr,atr,sl_dist,tp_dist
0,2026-02-17 02:00,SELL,False,❌ SL,-39.1,49486.30,49525.40,49042.61,11.35,True,192.91,39.10,443.69



  P&L Summary (passed signals only):
    ✅ TP total : +0.0 pts
    ❌ SL total : -39.1 pts
    Net P&L    : -39.1 pts


In [8]:
# ── Cell 8: Candlestick chart (Plotly) ────────────────────────────────────────
_chart_p = {
    **P,
    'ENTRY_LINE_BARS': ENTRY_LINE_BARS,
    'SHOW_REJECTED':   SHOW_REJECTED_SIGNALS,
    'SHOW_MA':         SHOW_MA_LINE,
    'SHOW_ENTRY_LINES': SHOW_ENTRY_LINES,
}
fig = build_reversal_chart(
    df_scan, signals_df, cfg,
    SCAN_SYMBOL, _chart_p,
    us_filtered_symbols=US_FILTERED_SYMBOLS,
)
fig.show()

In [9]:
# ── Cell 9: Interactive Reversal Scanner ──────────────────────────────────────
import ipywidgets as widgets
from IPython.display import HTML, clear_output

# ── Controls ─────────────────────────────────────────────────────────────────
w_symbol = widgets.Dropdown(
    options=list(TARGET_SYMBOLS.keys()), value=SCAN_SYMBOL,
    description='Symbol:', style={'description_width': 'auto'},
    layout=widgets.Layout(width='200px'),
)
w_nbars = widgets.BoundedIntText(
    value=N_BARS, min=30, max=500, step=10,
    description='N Bars:', style={'description_width': 'auto'},
    layout=widgets.Layout(width='145px'),
)
w_minrr = widgets.FloatSlider(
    value=MIN_RR, min=0.5, max=5.0, step=0.05,
    description='Min R:R:', readout_format='.2f',
    style={'description_width': '70px'}, layout=widgets.Layout(width='320px'),
)
w_ktp = widgets.FloatSlider(
    value=KTP, min=0.5, max=6.0, step=0.1,
    description='kTP:', readout_format='.2f',
    style={'description_width': '70px'}, layout=widgets.Layout(width='320px'),
)
w_rejected = widgets.Checkbox(value=True,  description='Show Rejected')
w_ma       = widgets.Checkbox(value=True,  description='Show MA')
w_entry    = widgets.Checkbox(value=True,  description='Show Entry/SL/TP')
w_btn      = widgets.Button(
    description='▶ Run Scan', button_style='success',
    layout=widgets.Layout(width='110px', height='34px'),
)
w_status = widgets.Label(value='')
w_out    = widgets.Output()


def _run(_=None):
    global SCAN_SYMBOL, N_BARS, SHOW_REJECTED_SIGNALS, SHOW_MA_LINE, SHOW_ENTRY_LINES
    global KTP, MIN_RR, P
    SCAN_SYMBOL           = w_symbol.value
    N_BARS                = w_nbars.value
    SHOW_REJECTED_SIGNALS = w_rejected.value
    SHOW_MA_LINE          = w_ma.value
    SHOW_ENTRY_LINES      = w_entry.value
    KTP                   = w_ktp.value
    MIN_RR                = w_minrr.value
    P = {**get_indicator_params(), 'KTP': KTP, 'MIN_RR': MIN_RR}

    w_status.value = f'⏳ Loading {SCAN_SYMBOL}...'

    with w_out:
        clear_output(wait=True)
        try:
            # ── Load & scan via reversal pipeline ─────────────────────────────
            result = run_reversal_scan(SCAN_SYMBOL, N_BARS, P)
            df_sc  = result['df_scan']
            sigs   = result['signals_df']
            cfg_i  = result['cfg']
            stats  = calc_reversal_stats(sigs)
            x_val  = cfg_i['x']

            # Add diagnostic columns
            if not sigs.empty:
                sigs = sigs.copy()
                sigs['C1_Candle']   = sigs['direction'].map({'BUY': 'Bull ✓', 'SELL': 'Bear ✓'})
                sigs['C2_MA_Cross'] = sigs['direction'].map({'BUY': '↑ Cross ✓', 'SELL': '↓ Cross ✓'})
                sigs['C3_MACD']     = sigs.apply(lambda r: f'{r["macd_h"]:+.1f} ✓', axis=1)
                sigs['C4_US30']     = sigs['us30_filtered'].map({True: '✓', False: '—'})
                sigs['C5_RR']       = sigs.apply(
                    lambda r: f'{r["rr"]:.2f} ✓' if r['pass_rr'] else f'{r["rr"]:.2f} ✗', axis=1)

            outcome_html = ''
            if stats['n_pass'] > 0:
                outcome_html = (
                    f'<span style="color:{SIGNAL["tp"]}">TP ✅ <b>{stats["n_tp"]}</b> ({stats["win_pct"]:.0f}%)</span>'
                    f'  <span style="color:{SIGNAL["sl"]}">SL ❌ <b>{stats["n_sl"]}</b></span>'
                    f'  <span style="color:#FFB347">Đảo 🔄 <b>{stats["n_reversed"]}</b></span>'
                )
                if stats['n_open']:
                    outcome_html += f'  <span style="color:#888">Mở: <b>{stats["n_open"]}</b></span>'

            # ── Summary bar ───────────────────────────────────────────────────
            display(HTML(f'''
<div style="margin:4px 0 0 0; padding:8px 14px; background:{DARK["panel"]};
            border-left:3px solid #FFB347; font-family:monospace;
            color:{DARK["text"]}; font-size:13px; display:flex; gap:20px; flex-wrap:wrap;">
  <span><b>{SCAN_SYMBOL}</b> &nbsp;·&nbsp; {N_BARS} bars H4 &nbsp;·&nbsp;
    <b style="color:#FFB347">Reversal Scanner</b></span>
  <span>Total: <b>{stats["n_total"]}</b>
    &nbsp;(Reversal: <b style="color:#FFB347">{stats["n_reversal_signals"]}</b>)</span>
  <span style="color:{SIGNAL["buy"]}">Pass R:R≥{MIN_RR:.2f}: <b>{stats["n_pass"]}</b>
    &nbsp;(BUY {stats["n_buy"]} / SELL {stats["n_sell"]})  avg R:R <b>{stats["avg_rr"]}</b></span>
  <span style="color:#888">Rejected: <b>{stats["n_rejected"]}</b></span>
  {outcome_html}
</div>
'''))

            # ── Công thức tính (hiển thị theo symbol đang chọn) ───────────────
            sym_label = cfg_i['label']
            display(HTML(f'''
<div style="margin:0 0 14px 0; padding:9px 16px; background:#111827;
            border-left:3px solid #2d4a6b; border-bottom:1px solid #1e2d40;
            font-family:monospace; font-size:12px; color:#bbb; line-height:2.0;">
  <span style="color:#607090; font-size:11px; text-transform:uppercase;
               letter-spacing:.07em;">📐 Công thức tính — {sym_label}
    &nbsp;(x={x_val} &nbsp;|&nbsp; kTP={KTP:.1f} &nbsp;|&nbsp; ATR period={P["ATR_PERIOD"]})</span><br>
  <span style="color:{SIGNAL["buy"]}; font-weight:bold;">BUY &nbsp;</span>
  &nbsp;Entry = High + <b>{x_val}</b>
  &nbsp;│&nbsp; SL = Low &minus; <b>{x_val}</b>
  &nbsp;│&nbsp; TP = Entry + <b>{KTP:.1f}</b> &times; ATR<br>
  <span style="color:{SIGNAL["sell"]}; font-weight:bold;">SELL</span>
  &nbsp;Entry = Low &minus; <b>{x_val}</b>
  &nbsp;│&nbsp; SL = High + <b>{x_val}</b>
  &nbsp;│&nbsp; TP = Entry &minus; <b>{KTP:.1f}</b> &times; ATR<br>
  <span style="color:#FFB347; font-weight:bold;">🔄 REVERSAL</span>
  &nbsp;Signal ngược chiều → đóng lệnh cũ (outcome=Reversed) → mở lệnh mới ngay<br>
  <span style="color:#505060; font-size:11px;">
    sl_dist = High &minus; Low + 2&times;{x_val}
    &nbsp;&middot;&nbsp; tp_dist = {KTP:.1f} &times; ATR
    &nbsp;&middot;&nbsp; R:R = tp_dist &divide; sl_dist
    &nbsp;&ge; <b>{MIN_RR:.2f}</b> để pass
  </span>
</div>
'''))

            # ── Signal table ─────────────────────────────────────────────────
            if not sigs.empty:
                sigs = _add_result_pnl(sigs)
                dcols = ['bar_time', 'direction', 'is_reversal', 'result', 'pnl_pts',
                         'C1_Candle', 'C2_MA_Cross', 'C3_MACD', 'C4_US30', 'C5_RR',
                         'entry', 'sl', 'tp', 'atr', 'sl_dist', 'tp_dist']
                tbl = sigs[dcols].copy()
                tbl['bar_time'] = tbl['bar_time'].dt.strftime('%Y-%m-%d %H:%M')

                cond_cols = ['C1_Candle', 'C2_MA_Cross', 'C3_MACD', 'C4_US30', 'C5_RR']
                display(tbl.style
                    .apply(style_reversal_row, axis=1)
                    .map(lambda v: (f'color:{SIGNAL["tp"]};font-weight:bold' if 'TP' in str(v)
                                    else (f'color:{SIGNAL["sl"]};font-weight:bold' if 'SL' in str(v)
                                          else ('color:#FFB347;font-weight:bold' if 'Đảo' in str(v)
                                                else ''))),
                         subset=['result'])
                    .map(lambda v: (f'color:{SIGNAL["tp"]};font-weight:bold' if v > 0
                                    else (f'color:{SIGNAL["sl"]};font-weight:bold' if v < 0
                                          else 'color:#555')),
                         subset=['pnl_pts'])
                    .map(lambda v: f'color:{SIGNAL["buy"]};font-weight:bold' if '✓' in str(v)
                         else (f'color:{SIGNAL["sl"]};font-weight:bold' if '✗' in str(v) else 'color:#555'),
                         subset=cond_cols)
                    .format({**NUM_FMT, 'pnl_pts': '{:+.1f}'})
                )
            else:
                print('Không có tín hiệu nào trong khoảng này.')

            # ── Chart section header ──────────────────────────────────────────
            display(HTML(f'''
<div style="margin:22px 0 0 0; padding:7px 16px; background:#0d1117;
            border-top:1px solid #30363d; border-bottom:1px solid #30363d;
            font-family:monospace; font-size:12px; color:#8b949e; letter-spacing:.05em;">
  📊 &nbsp;<b style="color:#c9d1d9;">REVERSAL CANDLESTICK CHART</b>
  &nbsp;·&nbsp; {sym_label}
  &nbsp;·&nbsp; {N_BARS} bars H4
  &nbsp;·&nbsp; kTP={KTP:.1f}
  &nbsp;·&nbsp; Min R:R={MIN_RR:.2f}
</div>
'''))

            # ── Chart ─────────────────────────────────────────────────────────
            _p = {
                **P,
                'ENTRY_LINE_BARS':  ENTRY_LINE_BARS,
                'SHOW_REJECTED':    SHOW_REJECTED_SIGNALS,
                'SHOW_MA':          SHOW_MA_LINE,
                'SHOW_ENTRY_LINES': SHOW_ENTRY_LINES,
            }
            fig = build_reversal_chart(df_sc, sigs, cfg_i, SCAN_SYMBOL, _p, US_FILTERED_SYMBOLS)
            display(fig)

            w_status.value = (
                f'✓ {SCAN_SYMBOL}  |  {stats["n_pass"]} pass  '
                f'TP={stats["n_tp"]} SL={stats["n_sl"]} Rev={stats["n_reversed"]}  '
                f'avg R:R={stats["avg_rr"]}'
            )

        except Exception as e:
            import traceback
            traceback.print_exc()
            w_status.value = f'⚠ Lỗi: {e}'


w_symbol.observe(lambda c: _run() if c['name'] == 'value' else None, names='value')
w_btn.on_click(_run)

display(widgets.VBox([
    widgets.HBox([w_symbol, w_nbars,
                  widgets.VBox([w_minrr, w_ktp]),
                  w_btn, w_status],
                 layout=widgets.Layout(align_items='center', gap='12px')),
    widgets.HBox([w_rejected, w_ma, w_entry]),
    w_out,
]))

_run()